# Udaipur Smart City — ML training pipeline

End-to-end training for three production models used by the FastAPI backend:

| Task | Target | Production features (must match `ml_service.py`) |
|------|--------|--------------------------------------------------|
| Traffic | `high_congestion` | `hour`, `day_enc`, `junction_enc`, `weather_enc`, `vehicles` |
| Waste | `overflow_risk` | `area`, `day_of_week`, `population_density`, `last_collection_days`, `bin_fill_pct` |
| Emergency | `high_risk` | `zone`, `hour`, `day_of_week`, `weather`, `road_condition` |

**Why tree ensembles?** Tabular municipal data with mixed numeric/categorical encodings; Random Forest, Gradient Boosting, and XGBoost handle non-linearities and class imbalance (via `class_weight`, `sample_weight`, and `scale_pos_weight`) without manual feature engineering beyond scaling.

**Why `StandardScaler` inside a `Pipeline`?** The hackathon spec requires scaled inputs; trees are scale-invariant, but a single `Pipeline` keeps preprocessing and the estimator together so `joblib` loads one artifact and `predict` matches training semantics.

**Leakage avoided:** We drop `congestion_score` (derived from same traffic dynamics as the label), `risk_score` / `incident_count` where they encode the outcome, and we do **not** use `bin_fill_pct` as a dropped target proxy for waste—we **keep** `bin_fill_pct` because it is the live sensor input the API receives at inference time (same as production).

## 0. Environment and paths

Run this notebook with the project virtualenv activated (`pip install -r requirements.txt`).  
Working directory can be either the repo `smart-city-allocation/` or `smart-city-allocation/notebooks/` — paths are resolved automatically.

In [ ]:
import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)


def resolve_project_paths():
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "traffic_clean.csv").exists():
        project_root = cwd
    elif (cwd.parent / "data" / "traffic_clean.csv").exists():
        project_root = cwd.parent
    else:
        raise FileNotFoundError(
            f"Could not find data/traffic_clean.csv from cwd={cwd}"
        )
    data_dir = project_root / "data"
    plot_dir = project_root / "notebooks" / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    return project_root, data_dir, plot_dir


PROJECT_ROOT, DATA_DIR, PLOT_DIR = resolve_project_paths()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PLOT_DIR:", PLOT_DIR)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load data

In [ ]:
traffic_df = pd.read_csv(DATA_DIR / "traffic_clean.csv")
waste_df = pd.read_csv(DATA_DIR / "waste_clean.csv")
emergency_df = pd.read_csv(DATA_DIR / "emergency_clean.csv")

datasets = {
    "traffic": (traffic_df, "high_congestion"),
    "waste": (waste_df, "overflow_risk"),
    "emergency": (emergency_df, "high_risk"),
}

## 2. EDA — shape, nulls, class balance

**Decision:** Stratified splits and class-weighted / weighted / XGBoost `scale_pos_weight` are justified when the positive class is rare (traffic ~4%, emergency ~8%).

In [ ]:
for name, (df, target) in datasets.items():
    print(f"\n{'='*60}\n{name.upper()}\n{'='*60}")
    print("Shape:", df.shape)
    print("Null counts:\n", df.isnull().sum())
    print("\nTarget counts:")
    print(df[target].value_counts())
    print("\nTarget proportion:")
    print(df[target].value_counts(normalize=True))
    print("\nNumeric describe():")
    print(df.describe().T)

## 3. Correlation heatmaps (numeric columns only)

**Decision:** Heatmaps guide feature selection; we still drop leakage columns even if correlation with the target is high.

In [ ]:
for name, (df, target) in datasets.items():
    num = df.select_dtypes(include=[np.number])
    plt.figure(figsize=(12, 9))
    corr = num.corr()
    sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, fmt=".2f")
    plt.title(f"Correlation — {name}")
    plt.tight_layout()
    out = PLOT_DIR / f"{name}_correlation.png"
    plt.savefig(out, dpi=150)
    plt.show()
    print("Saved", out)

## 4. Build feature matrices (names aligned with API)

- **Traffic:** Rename `junction`→`junction_enc`, `day_of_week`→`day_enc`, `weather`→`weather_enc`. Drop `temperature_c`, `congestion_score` (leakage / not used in API row).
- **Waste:** Keep `bin_fill_pct` — it is the operational input at prediction time.
- **Emergency:** Drop `incident_count` and `risk_score` (outcome-like); keep inputs available to `predict_emergency_risk(...)` in the backend.

In [ ]:
def build_traffic_xy(df: pd.DataFrame):
    d = df.rename(
        columns={
            "junction": "junction_enc",
            "day_of_week": "day_enc",
            "weather": "weather_enc",
        }
    )
    feature_cols = [
        "hour",
        "day_enc",
        "junction_enc",
        "weather_enc",
        "vehicles",
    ]
    X = d[feature_cols].copy()
    y = d["high_congestion"].astype(int)
    return X, y


def build_waste_xy(df: pd.DataFrame):
    feature_cols = [
        "area",
        "day_of_week",
        "population_density",
        "last_collection_days",
        "bin_fill_pct",
    ]
    X = df[feature_cols].copy()
    y = df["overflow_risk"].astype(int)
    return X, y


def build_emergency_xy(df: pd.DataFrame):
    feature_cols = ["zone", "hour", "day_of_week", "weather", "road_condition"]
    X = df[feature_cols].copy()
    y = df["high_risk"].astype(int)
    return X, y


XY_BUILDERS = {
    "traffic": build_traffic_xy,
    "waste": build_waste_xy,
    "emergency": build_emergency_xy,
}

for task, fn in XY_BUILDERS.items():
    X, y = fn(datasets[task][0])
    print(task, "X shape", X.shape, "columns", list(X.columns))

## 5. Preprocessing — label encoding (if needed) + stratified 80/20

Clean CSVs are already integer-coded. If any column were string, `LabelEncoder` would be applied per column.  
We then fit a column-wise `StandardScaler` via `ColumnTransformer` + `Pipeline` so the saved model applies the same transform at inference.

In [ ]:
def maybe_label_encode(X: pd.DataFrame):
    X = X.copy()
    encoders = {}
    for col in X.columns:
        if X[col].dtype == object or str(X[col].dtype).startswith("category"):
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            encoders[col] = le
    return X, encoders


def make_preprocessor(feature_names):
    return ColumnTransformer(
        [("scale", StandardScaler(), feature_names)],
        remainder="drop",
    )


def stratified_split_and_preprocess(task: str):
    df, _ = datasets[task]
    X, y = XY_BUILDERS[task](df)
    X, _ = maybe_label_encode(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=SEED,
        stratify=y,
    )
    return X_train, X_test, y_train, y_test, list(X.columns)


splits = {t: stratified_split_and_preprocess(t) for t in XY_BUILDERS.keys()}
for t, (Xtr, Xte, _, _, cols) in splits.items():
    print(t, "train", Xtr.shape, "test", Xte.shape, "features", cols)

## 6. Train three model families + 5-fold CV + test metrics

Models:
- **RandomForestClassifier** — `n_estimators=200`, `class_weight='balanced'`.
- **GradientBoostingClassifier** — no built-in `class_weight`; we use `sample_weight='balanced'` on `fit`.
- **XGBClassifier** — `scale_pos_weight = N_negative / N_positive` on the **training** fold.

Each estimator is wrapped in `Pipeline([('prep', ColumnTransformer...), ('clf', estimator)])`.

We select the **best test ROC-AUC** among the three (tie-break: higher CV mean ROC-AUC).

In [ ]:
def pos_class_weight(y_train):
    n_pos = (y_train == 1).sum()
    n_neg = (y_train == 0).sum()
    if n_pos == 0:
        return 1.0
    return float(n_neg) / float(n_pos)


def build_estimators(y_train):
    spw = pos_class_weight(y_train)
    sw = compute_sample_weight("balanced", y_train)
    return {
        "RandomForest": (
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1,
            ),
            {},
        ),
        "GradientBoosting": (
            GradientBoostingClassifier(
                n_estimators=200,
                learning_rate=0.08,
                max_depth=5,
                random_state=SEED,
            ),
            {"sample_weight": sw},
        ),
        "XGBoost": (
            XGBClassifier(
                n_estimators=200,
                max_depth=5,
                learning_rate=0.08,
                scale_pos_weight=spw,
                random_state=SEED,
                eval_metric="logloss",
                use_label_encoder=False,
            ),
            {},
        ),
    }


def train_and_evaluate_task(task_name: str):
    X_train, X_test, y_train, y_test, feature_names = splits[task_name]
    estimators = build_estimators(np.asarray(y_train))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    best_pipe = None
    best_auc = -1.0
    best_cv = -1.0
    best_label = None

    print(f"\n{'#'*70}\n# TASK: {task_name.upper()}\n{'#'*70}")

    for label, (est, fit_kwargs) in estimators.items():
        preprocessor = make_preprocessor(feature_names)
        pipe = Pipeline([("prep", preprocessor), ("clf", est)])
        cv_scores = cross_val_score(
            pipe,
            X_train,
            y_train,
            cv=skf,
            scoring="roc_auc",
            n_jobs=-1,
        )
        pipe.fit(X_train, y_train, **fit_kwargs)
        y_pred = pipe.predict(X_test)
        y_prob = pipe.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        print(f"\n--- {label} ---")
        print(f"CV ROC-AUC (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        print(f"Test ROC-AUC: {auc:.4f}")
        print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
        print(classification_report(y_test, y_pred, digits=4))

        cm_path = PLOT_DIR / f"{task_name}_{label}_confusion_matrix.png"
        plt.figure(figsize=(5, 4))
        sns.heatmap(
            confusion_matrix(y_test, y_pred),
            annot=True,
            fmt="d",
            cmap="Blues",
        )
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title(f"{task_name} — {label}")
        plt.tight_layout()
        plt.savefig(cm_path, dpi=150)
        plt.close()
        print("Saved", cm_path)

        better = auc > best_auc or (
            np.isclose(auc, best_auc) and cv_scores.mean() > best_cv
        )
        if better:
            best_auc = auc
            best_cv = cv_scores.mean()
            best_pipe = pipe
            best_label = label

    print(f"\n>>> Best for {task_name}: {best_label} (test ROC-AUC={best_auc:.4f})")

    clf = best_pipe.named_steps["clf"]
    if hasattr(clf, "feature_importances_"):
        fi = pd.DataFrame(
            {"feature": feature_names, "importance": clf.feature_importances_}
        ).sort_values("importance", ascending=False)
        plt.figure(figsize=(9, 5))
        sns.barplot(data=fi, x="importance", y="feature", palette="viridis")
        plt.title(f"Feature importance — {task_name} ({best_label})")
        plt.tight_layout()
        fi_path = PLOT_DIR / f"{task_name}_feature_importance.png"
        plt.savefig(fi_path, dpi=150)
        plt.show()
        print("Saved", fi_path)
    return best_pipe, best_label, best_auc


trained = {}
for t in ["traffic", "waste", "emergency"]:
    trained[t] = train_and_evaluate_task(t)

## 7. Save best models to project root

Files: `traffic_model.pkl`, `waste_model.pkl`, `emergency_model.pkl` next to `api/` — same paths as `ml_service.load_models()`.

In [ ]:
out_paths = {
    "traffic": PROJECT_ROOT / "traffic_model.pkl",
    "waste": PROJECT_ROOT / "waste_model.pkl",
    "emergency": PROJECT_ROOT / "emergency_model.pkl",
}

for task, pipe in trained.items():
    model_obj = pipe[0]  # (pipeline, label, auc)
    path = out_paths[task]
    joblib.dump(model_obj, path)
    print(f"Saved {task} -> {path}")

## 8. Smoke-test inference shape (matches API DataFrames)

Quick check that one row predicts without error using the same column names as `ml_service.py`.

In [ ]:
traffic_pipe = trained["traffic"][0]
waste_pipe = trained["waste"][0]
em_pipe = trained["emergency"][0]

Xt, _ = build_traffic_xy(traffic_df)
print("traffic predict", traffic_pipe.predict(Xt.head(1)))

Xw, _ = build_waste_xy(waste_df)
print("waste predict", waste_pipe.predict(Xw.head(1)))

Xe, _ = build_emergency_xy(emergency_df)
print("emergency predict", em_pipe.predict(Xe.head(1)))

print("Done.")